In [0]:
from src.config import *

# --- ADLS avain & Spark-konffi ---
access_key = dbutils.secrets.get("gs1-kv", "storage-access-key")
spark.conf.set(f"fs.azure.account.key.{ACCOUNT}.dfs.core.windows.net", access_key)

# --- Silver-polku ---
try:
    SILVER_PATH = DELTA_PRODUCTS
except NameError:
    CONTAINER = "datalake"
    SILVER_PATH = f"abfss://{CONTAINER}@{ACCOUNT}.dfs.core.windows.net/gs1/silver/catalogue_items"

print("Lähde:", SILVER_PATH)

# --- Lue vain Id + raw_json ---
df = (spark.read.format("delta").load(SILVER_PATH)
      .select("Id", "raw_json"))

import json

# Poimija: kerää KAIKKI urlit + polku + primary-heuristiikka yhdelle tuotteelle
def find_all_urls_with_primary(obj: dict):
    """
    Palauttaa listan osumia:
      { "path": <str>, "url": <str>, "primary": <bool>, "media_id": <optional> }
    Primary tulkitaan Trueksi jos samassa solmussa TAI peritysti:
      - IsPrimary / Primary / PrimaryFlag == truthy, TAI
      - AssetTypeCode viittaa pääkuvaan (esim. 'PRIMARY_IMAGE')
    URL-avaimet: Url/URL/Uri/URI/UniformResourceIdentifier (case-insensitive)
    """
    URL_KEYS = {"url", "uri", "uniformresourceidentifier"}
    PRIMARY_KEYS = {"isprimary", "primary", "primaryflag"}
    MEDIA_ID_KEYS = {"mediaitemid", "mediaid"}

    PRIMARY_TYPE_VALUES = {"primary_image", "primaryimage"}  # AssetTypeCode tms.
    hits = []

    def walk(node, path, inherited_primary=False, inherited_mediaid=None):
        if isinstance(node, dict):
            keys_lower = {k.lower(): k for k in node.keys()}

            # Päivitä primary/mediaid periytyviksi lipuiksi
            local_primary = inherited_primary or any(
                (k in keys_lower) and bool(node[keys_lower[k]])
                for k in PRIMARY_KEYS
            )
            # AssetTypeCode -> primary
            atc_key = keys_lower.get("assettypecode")
            if atc_key:
                val = str(node[atc_key]).strip().lower()
                if val in PRIMARY_TYPE_VALUES:
                    local_primary = True

            local_mediaid = inherited_mediaid
            for midk in MEDIA_ID_KEYS:
                if midk in keys_lower:
                    local_mediaid = node[keys_lower[midk]]
                    break

            # Kerää URL tällä tasolla
            for urlk in URL_KEYS:
                if urlk in keys_lower:
                    raw_key = keys_lower[urlk]
                    val = node.get(raw_key)
                    if isinstance(val, str) and val:
                        hits.append({
                            "path": f"{path}.{raw_key}".lstrip("."),
                            "url": val,
                            "primary": bool(local_primary),
                            "media_id": local_mediaid
                        })

            # Rekursio
            for k, v in node.items():
                walk(v, f"{path}.{k}" if path else k, local_primary, local_mediaid)

        elif isinstance(node, list):
            for v in node:
                walk(v, f"{path}[]" if path else "[]", inherited_primary, inherited_mediaid)

    walk(obj, "", False, None)
    return hits

# --- Etsi top N eri tuotetta, joissa on vähintään yksi URL ---
TARGET_COUNT = 10
MAX_CHECK = 200000  # turvaraja, ettei skannaa loputtomiin

examples = []            # lista: {"Id": ..., "hits": [...]}
seen_ids = set()
checked = 0

for row in df.toLocalIterator():  # streamaa partitiokerrallaan
    checked += 1
    pid = row["Id"]
    if pid in seen_ids:
        continue
    seen_ids.add(pid)

    try:
        obj = json.loads(row["raw_json"])
    except Exception:
        obj = {}

    hits = find_all_urls_with_primary(obj)
    if hits:
        # järjestä: primary=True ensin, sitten polun mukaan
        hits.sort(key=lambda x: (not x["primary"], x["path"]))
        examples.append({"Id": pid, "hits": hits})

    if len(examples) >= TARGET_COUNT or checked >= MAX_CHECK:
        break

    if checked % 5000 == 0:
        print(f"checked={checked:,}  found_products={len(examples)}")

print(f"\nValmista. Tarkistettu {checked:,} riviä, löytyi {len(examples)} tuotetta joilla vähintään 1 URL.\n")

# --- Tulosta top N tuotteet ja NIIDEN KAIKKI URLIT (primary-flag mukana) ---
for idx, ex in enumerate(examples, 1):
    print(f"[{idx}] Id={ex['Id']} — url-osumia: {len(ex['hits'])}")
    for j, h in enumerate(ex["hits"], 1):
        print(f"    ({j}) primary={str(h['primary']).ljust(5)}  path={h['path']}\n"
              f"         url={h['url']}\n"
              f"         media_id={h['media_id']}\n")


In [0]:
import json, re
from collections import Counter
from src.config import *

rx = re.compile(r"_([A-Za-z0-9]{4})\.", re.IGNORECASE)
cnt = Counter()

for r in (spark.read.format("delta").load(DELTA_PRODUCTS)
          .select("raw_json").limit(10000).toLocalIterator()):
    try:
        obj = json.loads(r["raw_json"])
        ti = (obj or {}).get("TradeItem") or {}
        headers = ((ti.get("ReferencedFileDetailInformationModule") or {}).get("ReferencedFileHeader"))
        if isinstance(headers, list):
            for h in headers:
                u = (h or {}).get("Url") or (h or {}).get("URL") or (h or {}).get("UniformResourceIdentifier")
                if isinstance(u, str):
                    m = rx.search(u)
                    if m:
                        cnt[m.group(1).upper()] += 1
    except Exception:
        pass

print("TOP-suffiksit:")
for s, n in cnt.most_common(15):
    print(f"{s}: {n}")


In [0]:
from src.config import *

import json, re
from collections import Counter, defaultdict

# --- URL-etsintäfunktio ---
def find_all_urls_with_primary(obj: dict):
    URL_KEYS = {"url", "uri", "uniformresourceidentifier"}
    PRIMARY_KEYS = {"isprimary", "primary", "primaryflag"}
    MEDIA_ID_KEYS = {"mediaitemid", "mediaid"}
    PRIMARY_TYPE_VALUES = {"primary_image", "primaryimage"}
    hits = []

    def walk(node, path, inherited_primary=False, inherited_mediaid=None):
        if isinstance(node, dict):
            keys_lower = {k.lower(): k for k in node.keys()}
            local_primary = inherited_primary or any(
                (k in keys_lower) and bool(node[keys_lower[k]])
                for k in PRIMARY_KEYS
            )
            atc_key = keys_lower.get("assettypecode")
            if atc_key:
                val = str(node[atc_key]).strip().lower()
                if val in PRIMARY_TYPE_VALUES:
                    local_primary = True

            local_mediaid = inherited_mediaid
            for midk in MEDIA_ID_KEYS:
                if midk in keys_lower:
                    local_mediaid = node[keys_lower[midk]]
                    break

            for urlk in URL_KEYS:
                if urlk in keys_lower:
                    val = node.get(keys_lower[urlk])
                    if isinstance(val, str) and val:
                        hits.append({
                            "path": f"{path}.{keys_lower[urlk]}".lstrip("."),
                            "url": val,
                            "primary": bool(local_primary),
                            "media_id": local_mediaid
                        })

            for k, v in node.items():
                walk(v, f"{path}.{k}" if path else k, local_primary, local_mediaid)

        elif isinstance(node, list):
            for v in node:
                walk(v, f"{path}[]" if path else "[]", inherited_primary, inherited_mediaid)

    walk(obj, "", False, None)
    return hits


# --- Asetukset ---
rx_suffix = re.compile(r"_([A-Za-z0-9]{4})\.", re.IGNORECASE)
MAX_CHECK = 10000  # voit nostaa halutessa
df = spark.read.format("delta").load(DELTA_PRODUCTS).select("Id", "raw_json")

checked = 0
for row in df.toLocalIterator():
    checked += 1
    pid = row["Id"]

    try:
        obj = json.loads(row["raw_json"])
    except Exception:
        continue

    hits = find_all_urls_with_primary(obj)
    if not hits:
        continue

    # --- Laske suffiksit ja ryhmittele ---
    suffix_counter = Counter()
    urls_by_suffix = defaultdict(list)

    for h in hits:
        url = h["url"]
        m = rx_suffix.search(url)
        suff = m.group(1).upper() if m else "NONE"
        suffix_counter[suff] += 1
        urls_by_suffix[suff].append(h)

    # --- Tulosta per tuote ---
    print(f"\nEAN={pid}")
    print(f"  URL-määrä: {len(hits)}")
    print(f"  Suffiksit: {', '.join([f'{s}({n})' for s, n in suffix_counter.items()])}")

    for suff, url_hits in urls_by_suffix.items():
        print(f"    [{suff}]")
        for h in url_hits[:3]:  # max 3 riviä per suffiksi tulostukseen
            print(f"        primary={str(h['primary']).ljust(5)}  url={h['url']}")

    if checked >= MAX_CHECK:
        break


In [0]:
import json, re
from collections import Counter, defaultdict

# --- 1) Apu: poimi kaikki urlit (sama logiikka kuin aiemmin) ---
def find_all_urls_with_primary(obj: dict):
    URL_KEYS = {
    "url", "uri", "uniformresourceidentifier",
    "fileurl", "httpurl", "httpsurl",
    "referencedfileuniformresourceidentifier",
}
    PRIMARY_KEYS = {"isprimary", "primary", "primaryflag"}
    MEDIA_ID_KEYS = {"mediaitemid", "mediaid"}
    PRIMARY_TYPE_VALUES = {"primary_image", "primaryimage"}
    hits = []

    def walk(node, path, inherited_primary=False, inherited_mediaid=None):
        if isinstance(node, dict):
            keys_lower = {k.lower(): k for k in node.keys()}

            local_primary = inherited_primary or any(
                (k in keys_lower) and bool(node[keys_lower[k]]) for k in PRIMARY_KEYS
            )
            atc_key = keys_lower.get("assettypecode")
            if atc_key:
                val = str(node[atc_key]).strip().lower()
                if val in PRIMARY_TYPE_VALUES:
                    local_primary = True

            local_mediaid = inherited_mediaid
            for midk in MEDIA_ID_KEYS:
                if midk in keys_lower:
                    local_mediaid = node[keys_lower[midk]]
                    break

            for urlk in URL_KEYS:
                if urlk in keys_lower:
                    val = node.get(keys_lower[urlk])
                    if isinstance(val, str) and val:
                        hits.append({
                            "path": f"{path}.{keys_lower[urlk]}".lstrip("."),
                            "url": val,
                            "primary": bool(local_primary),
                            "media_id": local_mediaid
                        })

            for k, v in node.items():
                walk(v, f"{path}.{k}" if path else k, local_primary, local_mediaid)

        elif isinstance(node, list):
            for v in node:
                walk(v, f"{path}[]" if path else "[]", inherited_primary, inherited_mediaid)

    walk(obj, "", False, None)
    return hits


# --- 2) Heuristiikat: suffiksit, resoluutio, tiedostotyyppi, polkuvihjeet ---
rx_suffix = re.compile(r"_([A-Za-z0-9]{4})\.", re.IGNORECASE)
rx_res = re.compile(r"(?:(?:_|-)(\d{3,5})[xX](\d{3,5}))|(?:[=/\-_](\d{3,5}))(?=[._-]?(?:px|w|h|$))")

SUFFIX_PRIORITY = {
    "C1C1": 120, "C1N1": 115, "A1C1": 110, "H1N1": 105, "A1N1": 102,
    "C1L1": 100, "C1R1": 98, "C1N0": 96, "A1C0": 94, "C1C0": 92,
    "H1C1": 90, "A1L1": 88, "C3C1": 86, "D1NE": 84, "A1N0": 82,
}
EXT_PRIORITY = {"jpg": 10, "jpeg": 10, "png": 8, "webp": 6, "tif": 5, "tiff": 5, "gif": 2}
PATH_HINTS = ("primary", "main", "hero")

def extract_suffix(url: str) -> str:
    m = rx_suffix.search(url)
    return m.group(1).upper() if m else "NONE"

def extract_resolution(url: str):
    m = rx_res.search(url)
    if not m:
        return (None, None)
    if m.group(1) and m.group(2):
        return (int(m.group(1)), int(m.group(2)))
    if m.group(3):
        v = int(m.group(3))
        return (v, v)
    return (None, None)

def ext_of(url: str) -> str:
    part = url.split("?")[0]
    if "." in part:
        return part.rsplit(".", 1)[1].lower()
    return ""

def score_url(url: str, suff: str) -> float:
    sfx = SUFFIX_PRIORITY.get(suff, 0)
    w, h = extract_resolution(url)
    res_score = 0.0
    if w and h:
        from math import log10
        res_score = log10(max(1, w*h))
    ext_score = EXT_PRIORITY.get(ext_of(url), 0)
    path_bonus = 5 if any(hint in url.lower() for hint in PATH_HINTS) else 0
    return sfx + res_score + ext_score + path_bonus


# --- 3) Lue data, valitse paras URL per EAN + kerää koonti ---
MAX_CHECK = 20000
df = spark.read.format("delta").load(DELTA_PRODUCTS).select("Id", "raw_json")

checked_total = 0               # montako EANia käytiin läpi (tarkistettiin)
products_with_urls = 0          # montako EANia, joille löytyi vähintään yksi URL
total_urls_found = 0            # kaikkien löydettyjen URLien määrä
hit_max_check = False

for row in df.toLocalIterator():
    checked_total += 1
    pid = row["Id"]

    try:
        obj = json.loads(row["raw_json"])
    except Exception:
        obj = {}

    hits = find_all_urls_with_primary(obj)
    if not hits:
        if checked_total >= MAX_CHECK:
            hit_max_check = True
            break
        continue

    products_with_urls += 1
    total_urls_found += len(hits)

    # Ryhmitä media_id:llä (tai yhteinen None-ryhmä)
    by_media = defaultdict(list)
    for h in hits:
        by_media[h["media_id"]].append(h)

    # Pisteytä kaikki urlit per ryhmä, valitse kunkin ryhmän paras
    group_bests = []
    suffix_counter = Counter()

    for mid, items in by_media.items():
        scored = []
        for h in items:
            suff = extract_suffix(h["url"])
            suffix_counter[suff] += 1
            scored.append({
                "url": h["url"],
                "primary": h["primary"],
                "suffix": suff,
                "score": score_url(h["url"], suff)
            })
        best_in_group = max(scored, key=lambda x: x["score"])
        group_bests.append((mid, best_in_group, sorted(scored, key=lambda x: -x["score"])[:3]))

    # Lopullinen paras (paras parhaiden joukosta)
    final_mid, final_best, top3_debug = max(group_bests, key=lambda t: t[1]["score"])

    # Tulostus per tuote (voit poistaa jos haluat vain koonnin)
    print(f"\nEAN={pid}")
    print(f"  URL-määrä: {len(hits)}  | media-ryhmiä: {len(by_media)}")
    print(f"  Suffiksit: {', '.join([f'{s}({n})' for s, n in suffix_counter.most_common()])}")
    print(f"  >> VALITTU: suffix={final_best['suffix']}  score={final_best['score']:.2f}  primary={final_best['primary']}")
    print(f"     url={final_best['url']}")
    print(f"  -- Top-3 pisteytettyä koko datasta debugiksi --")
    all_scored = []
    for _, best, top3 in group_bests:
        all_scored.extend(top3)
    all_scored.sort(key=lambda x: -x["score"])
    for i, cand in enumerate(all_scored[:3], 1):
        print(f"     ({i}) suffix={cand['suffix']} score={cand['score']:.2f} primary={cand['primary']} | {cand['url']}")

    if checked_total >= MAX_CHECK:
        hit_max_check = True
        break

# --- 4) Yhteenveto ---
pct = (products_with_urls / checked_total * 100) if checked_total else 0.0
print("\n=== YHTEENVETO ===")
print(f"Tarkistettu EANeja: {checked_total:,}")
print(f"EANeja, joille löytyi vähintään yksi URL: {products_with_urls:,} ({pct:.1f} %)")
print(f"Löydettyjen URLien kokonaismäärä: {total_urls_found:,}")
if hit_max_check:
    print(f"Huom: MAX_CHECK-raja ({MAX_CHECK:,}) saavutettiin — ajo pysäytettiin siihen.")


In [0]:
import json
from collections import Counter

URL_KEYS = {"url","uri","uniformresourceidentifier","fileurl","httpurl","httpsurl",
            "referencedfileuniformresourceidentifier"}

def has_url_rec(node) -> bool:
    if isinstance(node, dict):
        kl = {k.lower(): k for k in node.keys()}
        if any(k in kl and isinstance(node[kl[k]], str) and node[kl[k]]
               for k in URL_KEYS):
            return True
        return any(has_url_rec(v) for v in node.values())
    if isinstance(node, list):
        return any(has_url_rec(v) for v in node)
    return False

def get_descriptor(obj: dict) -> str:
    # UUSI: oikea polku koodiarvolle
    ti = (obj or {}).get("TradeItem") or {}
    code = ((ti.get("TradeItemUnitDescriptorCode") or {}).get("Value"))
    if isinstance(code, str) and code.strip():
        return code.strip()
    # fallbackit (joskus joissain dumpissa eri nimellä)
    for key in ("TradeItemUnitDescriptor","tradeItemUnitDescriptor",
                "TradeItemUnitDescriptorCode","tradeItemUnitDescriptorCode"):
        v = ti.get(key)
        if isinstance(v, dict) and isinstance(v.get("Value"), str):
            return v["Value"].strip()
        if isinstance(v, str) and v.strip():
            return v.strip()
    return "UNKNOWN"

total_by_desc = Counter()
with_url_by_desc = Counter()

df = spark.read.format("delta").load(DELTA_PRODUCTS).select("raw_json")

for r in df.toLocalIterator():
    try:
        obj = json.loads(r["raw_json"])
    except Exception:
        obj = {}
    desc = get_descriptor(obj)
    total_by_desc[desc] += 1
    if has_url_rec(obj):
        with_url_by_desc[desc] += 1

print("\n=== KUVA-KATTAVUUS DESCRIPTORIN MUKAAN (oma kuva) ===")
grand_total = sum(total_by_desc.values())
grand_with  = sum(with_url_by_desc.values())
for desc, n in total_by_desc.most_common():
    w = with_url_by_desc[desc]
    pct = (w/n*100) if n else 0
    print(f"- {desc:>18}: {w:>7}/{n:<7}  ({pct:5.1f}%)")
print(f"\nYhteensä: {grand_with}/{grand_total} ({(grand_with/grand_total*100 if grand_total else 0):.1f}%)")
